In [95]:
import json
import pandas as pd
import numpy as np

leader_files = ['old_data/65_team_info.csv', 'old_data/82_team_info.csv', 'old_data/852_team_info.csv']
leader_list = []
for f in leader_files:
    with open(f, 'r', encoding='UTF8') as file:
        leader_list.append(pd.DataFrame(pd.read_csv(file)))


In [96]:
team_files = ['old_data/team_members_HK.json', 'old_data/team_members_SG.json', 'old_data/team_members_KR.json']

team_list = []

for f in team_files:
    with open(f, "r", encoding="UTF8") as file:
        team_list.append(pd.DataFrame(json.load(file)['list']))

team_df = pd.concat(leader_list+team_list, ignore_index=True)  


In [97]:
mask = team_df['pids'].isna()
team_df.loc[mask, 'pids'] = "88888888"
team_df['pid'] = team_df["pids"].str.split(',').str[-1]
team_df = team_df.rename(columns={'teamName': 'username'}).drop(columns= ["personalInvestIsAuth"])

In [98]:
users_df = pd.read_json('data/user_info.json', encoding='utf8')
users_df = pd.concat([ users_df[['server_code', 'password']], pd.json_normalize(users_df['row']).drop(columns={'id','status', 'joinTime', 'password', 'updateTime', 'avatar', 'inviteNum', 'invitees', "mobile", "invitationCode", "areaCode", "loginIp", "loginTime", "birthday", "messagePd", "loginFailure", "pushClientId", "isTop", "risk"})], axis=1).rename(columns={'pUserId': 'pid', 'pIds': 'pids'})
users_df = users_df.drop_duplicates(subset=['userId', 'username'])

In [99]:
team_df = team_df.astype({"userId":"str", "pid":"str", "pids":"str", "idcardAuthStatus":"Int64", "username": "str"})
users_df = users_df.astype({"userId": "str", "password":"str", "pid":str})

number_userId_filter = pd.to_numeric(team_df['userId'], errors='coerce')
number_UID_team_df = team_df[number_userId_filter.notnull()]
masked_UID_team_df = team_df[~number_userId_filter.notnull()]

In [100]:
number_UID_team_df = number_UID_team_df.set_index(['userId', "username"])
users_df = users_df.set_index(['userId', "username"])
numUID_team_users_df = users_df.combine_first(number_UID_team_df)

In [101]:
invest_df = pd.read_json('data/invest_Auth.json')
invest_df = pd.json_normalize(invest_df[invest_df['row'].notna()]['row']).drop(columns=['rewardStatus', 'idCardSfz1', 'idCardSfz2', "idCard", "createTime", "id", "areaCode", "type", "isTuo", "realName"])
invest_df = invest_df.drop_duplicates(subset=['userId', "username"])
invest_df = invest_df.astype({"userId": "str"})

In [102]:
invest_df.loc[invest_df['userId']=="78968835", "username"]="01072351656"
invest_df = invest_df.set_index(["userId", "username"])

In [103]:
numUID_team_users_invest_df = invest_df.combine_first(numUID_team_users_df)
numUID_team_users_invest_df= numUID_team_users_invest_df.reset_index()

In [104]:
result = pd.concat([numUID_team_users_invest_df, masked_UID_team_df], axis=0, ignore_index=True)
result = result.astype({"status": "Int64", "mStepNumber":"Int64", "motionLevel":"Int64", "pid": "str", "totalMStepNumber": "Int64", "level":"Int64", "isSalesman": "boolean", "gender": "boolean", "idcardAuthStatus":"Int64", "listingSum":"Int64", "salesmanUserId": "Int64", "videoTotalAppreciateNum":"Int64", "isVideoUser":"boolean", "rank1":"Int64", "isTuo":"boolean","isBusiness":"boolean", "isRestockUser":"boolean", "id":"Int64", "teamTotalNumber":"Int64", "teamLevel": "Int64", "motionTeamLevel":"Int64", "teamTotalStep":"Int64","personalTotalStep":"Int64", "authGcm":"Int64"})

In [105]:
cols = ["createTime", "updateTime"]
result[cols] = result[cols].apply(lambda s: pd.to_datetime(s, unit='ms', utc=True)
                                     .dt.tz_convert('Asia/Hong_Kong')
                                     .dt.strftime('%Y%m%d %H:%M'))


In [106]:
result['MLM_Level'] = result['pids'].str.count(",").astype("Int64")

In [ ]:
masked_ids = []
#undone_rows = pd.DataFrame()
unmatched_candidates = pd.DataFrame()
mapper = pd.Series()
gv = {"masked_rows":pd.DataFrame(), "candidate_table":pd.DataFrame()}
def Map_Filter (Candidate_Indexer, Mapkey):
    mapper = gv["candidate_table"].set_index(Candidate_Indexer)
    gv["masked_rows"]["map_key"] = Mapkey
    gv["masked_rows"]["userId"] = gv["masked_rows"]['map_key'].map(mapper["aid"])
    gv["candidate_table"] = gv["candidate_table"][~gv["candidate_table"]['aid'].isin(gv["masked_rows"]["userId"])]
    assign_rows, gv["masked_rows"] = gv["masked_rows"][~(m:=gv["masked_rows"]['userId'].isna())], gv["masked_rows"][m]
    result.loc[assign_rows.index, "userId"] = assign_rows["userId"]

for level in reversed(range(1, 15)):
    gv["masked_rows"] = Top3Levels[(result['MLM_Level']==level) & (result['teamTotalNumber']> 0) & (Top3Levels['userId'] == "****")].assign(IntegralDiff=lambda s:round(s["teamTotalIntegral"]-s["personalTotalIntegral"], 2))[['pids','teamTotalNumber', "IntegralDiff"]]
    visible_ids = Top3Levels[(Top3Levels['MLM_Level']==level) & (Top3Levels['userId'] != "****")]['userId']
    masked_id_candidates = Top3Levels.loc[lambda s: (s['MLM_Level']==level+1) & (~s['pid'].isin(visible_ids)), "pid"].unique()
    masked_id_candidates_regex_str = f'({"|".join(masked_id_candidates)})'
    masked_ids.extend(masked_id_candidates)
    gv["candidate_table"] = Top3Levels['pids'].str.extract("(^.*),"+masked_id_candidates_regex_str, expand=False).rename(columns={0:"ancester_path", 1:"aid"}).value_counts().reset_index()
    for uid in masked_id_candidates:
        gv["candidate_table"].loc[gv["candidate_table"]["aid"] == uid, "downlineIntegral"] = round(Top3Levels[result['pids'].str.contains(uid)]["personalTotalIntegral"].sum(), 2) 

    gv["masked_rows"],lookalike_rows = (gv["masked_rows"][~(m := gv["masked_rows"][["pids", "teamTotalNumber", "IntegralDiff"]].duplicated(keep=False))], gv["masked_rows"][m])

    gv["candidate_table"], lookalike_candidates = (gv["candidate_table"][~(m:=gv["candidate_table"].duplicated(subset=["ancester_path", "count", "downlineIntegral"], keep=False))], gv["candidate_table"][m])
    unmatched_candidates = pd.concat([lookalike_candidates, unmatched_candidates])
    
    Map_Filter(["ancester_path", "count", "downlineIntegral"], list(zip( gv["masked_rows"]["pids"], gv["masked_rows"]["teamTotalNumber"], gv["masked_rows"]["IntegralDiff"])))

    if(gv["masked_rows"].shape[0]>0):
        Map_Filter(["ancester_path", "count"], list(zip(gv["masked_rows"]["pids"], gv["masked_rows"]["teamTotalNumber"])))

    if(gv["masked_rows"].shape[0]>0):
        Map_Filter("ancester_path", gv["masked_rows"]["pids"])
    

In [114]:
result.loc[46893, "userId"] = "32333187"
result.loc[47109, "userId"] = "72510007"
result.loc[46790,"userId"] = "40885376"
result.loc[46680,"userId"] = "49596546"
result.loc[47013,"userId"] = "32372489"
result.loc[47035,"userId"] = "60100351"

In [117]:
username_result = result[(result["pid"].isin(masked_ids)) & (result["pUsername"].notna())][['pid','pUsername']].drop_duplicates(subset=['pid', 'pUsername'], keep='first').set_index("pid")

In [118]:
mask = result['userId'].isin(username_result.index)
result.loc[mask, 'username'] = result.loc[mask, "userId"].map(username_result['pUsername'])

In [119]:
displaynames= pd.read_excel("whatsapp_group_members.xlsx")

In [120]:
displaynames = displaynames.set_index("MatchNum")['display_name']

In [121]:
result['display_name'] = result['username'].map(displaynames).combine_first(result['realName'])

In [122]:
result = result.reindex(columns=['userId', 'username', 'countryRegion', 'id', 'pid', 'pids', 'pUsername', 'MLM_Level',
               'display_name', 'realName', 'nickName', 'authGcm', 'createTime', 'updateTime', 'email', 'avatar', 'bio', 'gender',
               'joinIp', 'lang','idCard', 'idcardAuthStatus', 'status','remark', 'level', 'sfz1', 'sfz2','server_code', 
               'teamTotalNumber', 'teamLevel', 'teamTotalCoffee', 'teamTotalIntegral','teamTotalInvest', 'teamTotalRecharge','teamTotalStep', 'teamTotalWithdrawal', 
               'totalInvestBank','totalInvestUsdt', 'totalMStepNumber', 'totalRechargeBank','totalRechargeUsdt', 'totalWaitCapitalBank', 'totalWaitCapitalUsdt','totalWaitInterestBank', 'totalWaitInterestUsdt', 'totalWithdrawalBank','totalWithdrawalUsdt', 
                'personalTotalInvest', 'personalTotalRecharge','personalTotalIntegral', 'personalTotalStep', 'personalTotalWithdrawal', 'personalKickback', 'personalTotalCoffee',
                'mIntegralMoney', 'mStepNumber', 'motionLevel', 'motionTeamLevel','coffeeMoney',
                 'salesmanUserId', 'operator', 'operatorBy', 
                'userLevelRate', 'rank1','listingSum', 
                'isBusiness','isRestockUser', 'isSalesman', 'isTuo', 'isVideoUser', 
                'wCurrencyMoneyBank','wCurrencyMoneyUsdt',
                'password','googleAuthSecret',  'payPassword',
                'videoTotalAppreciateNum'] )

In [119]:
from collections import Counter

visit_count = Counter()

def walk(node, path=None):
    if path is None:
        path = set()

    node_id = id(node)

    # detect cycles
    if node_id in path:
        print("CYCLE DETECTED")
        print(node)
        return

    visit_count[node_id] += 1

    new_path = path | {node_id}

    for child in node.get("children", []):
        walk(child, new_path)

# traverse all roots
for root in roots:
    walk(root)

print(f"Rows in df      : {len(Top3Levels)}")
print(f"Visited nodes   : {sum(visit_count.values())}")
print(f"Unique nodes    : {len(visit_count)}")

duplicates = {
    node_id: count
    for node_id, count in visit_count.items()
    if count > 1
}

print(f"Duplicate nodes : {len(duplicates)}")

Rows in df      : 688
Visited nodes   : 695
Unique nodes    : 637
Duplicate nodes : 8


In [ ]:
import pandas as pd
import json
import re

# count digits in a phone number / username
def digit_count(value):
    if pd.isna(value):
        return 0

    return sum(c.isdigit() for c in str(value))


# remove unwanted fields
def clean_record(record):
    cleaned = {}

    for k, v in record.items():

        # remove NaN / None
        if pd.isna(v):
            continue

        # remove empty strings and whitespace-only strings
        if isinstance(v, str) and v.strip() == "":
            continue

        # remove numeric zero values
        if isinstance(v, (int, float)) and v == 0:
            continue

        cleaned[k] = v

    return cleaned


# create unique internal row id
Top3Levels = result.reset_index(drop=True)
result["_row_id"] = result.index

# create node objects
nodes = {}

for _, row in result.iterrows():

    node = clean_record(row.to_dict())

    node.pop("_row_id", None)

    node["children"] = []

    nodes[row["_row_id"]] = node


roots = []

# build hierarchy
for _, child_row in result.iterrows():

    child_node = nodes[child_row["_row_id"]]

    # all candidate parents
    candidates = result[
        result["userId"].astype(str) == str(child_row["pid"])
    ]

    if len(candidates) == 0:

        # root node
        roots.append(child_node)

    elif len(candidates) == 1:

        # only one possible parent
        parent_row_id = candidates.iloc[0]["_row_id"]

        nodes[parent_row_id]["children"].append(
            child_node
        )

    else:

        # duplicate userId found
        # choose parent whose username has same digit count
        child_digits = digit_count(
            child_row.get("username")
        )

        candidates = candidates[
            candidates["username"]
            .map(digit_count)
            .eq(child_digits)
        ]

        if len(candidates) == 1:

            parent_row_id = candidates.iloc[0]["_row_id"]

            nodes[parent_row_id]["children"].append(
                child_node
            )

        else:

            print(
                f"Unable to uniquely identify parent "
                f"for userId={child_row['userId']} "
                f"pid={child_row['pid']}"
            )

            roots.append(child_node)


# save pretty JSON
with open(
    "full_hierarchy.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        roots,
        f,
        ensure_ascii=False,
        indent=2
    )


print(
    f"Created {len(roots)} root nodes"
)

Created 3 root nodes


In [124]:
result.to_csv("final_out.csv")

In [125]:
Top3Levels = result[result['MLM_Level']< 4][['userId','pid', 'username', 'display_name', 'realName', 
                                           'rank1','status','authGcm', 'avatar', 'bio', 'countryRegion', 'lang','gender', 
                                            'createTime', 'updateTime', 'joinIp', 'idcardAuthStatus', 
                                            'isTuo', 'isSalesman','salesmanUserId','level','userLevelRate',
                                            'mStepNumber', 'motionLevel', 'motionTeamLevel',
                                            'personalKickback', 'personalTotalInvest', 'personalTotalRecharge','personalTotalStep', 'personalTotalWithdrawal',
                                            'teamLevel', 'teamTotalIntegral','teamTotalInvest', 'teamTotalNumber', 'teamTotalStep', 'teamTotalWithdrawal', 
                                            'totalInvestBank','totalInvestUsdt', 'totalMStepNumber', 'totalRechargeBank','totalRechargeUsdt', 'totalWaitCapitalBank', 'totalWaitCapitalUsdt','totalWaitInterestBank', 'totalWaitInterestUsdt', 'totalWithdrawalBank','totalWithdrawalUsdt',
                                            'isVideoUser','videoTotalAppreciateNum']]

In [126]:
import pandas as pd
import json
import re

# count digits in a phone number / username
def digit_count(value):
    if pd.isna(value):
        return 0

    return sum(c.isdigit() for c in str(value))


# remove unwanted fields
def clean_record(record):
    cleaned = {}

    for k, v in record.items():

        # remove NaN / None
        if pd.isna(v):
            continue

        # remove empty strings and whitespace-only strings
        if isinstance(v, str) and v.strip() == "":
            continue

        # remove numeric zero values
        if isinstance(v, (int, float)) and v == 0:
            continue

        cleaned[k] = v

    return cleaned


# create unique internal row id
Top3Levels = Top3Levels.reset_index(drop=True)
Top3Levels["_row_id"] = Top3Levels.index

# create node objects
nodes = {}

for _, row in Top3Levels.iterrows():

    node = clean_record(row.to_dict())

    node.pop("_row_id", None)

    node["children"] = []

    nodes[row["_row_id"]] = node


roots = []

# build hierarchy
for _, child_row in Top3Levels.iterrows():

    child_node = nodes[child_row["_row_id"]]

    # all candidate parents
    candidates = Top3Levels[
        Top3Levels["userId"].astype(str) == str(child_row["pid"])
    ]

    if len(candidates) == 0:

        # root node
        roots.append(child_node)

    elif len(candidates) == 1:

        # only one possible parent
        parent_row_id = candidates.iloc[0]["_row_id"]

        nodes[parent_row_id]["children"].append(
            child_node
        )

    else:

        # duplicate userId found
        # choose parent whose username has same digit count
        child_digits = digit_count(
            child_row.get("username")
        )

        candidates = candidates[
            candidates["username"]
            .map(digit_count)
            .eq(child_digits)
        ]

        if len(candidates) == 1:

            parent_row_id = candidates.iloc[0]["_row_id"]

            nodes[parent_row_id]["children"].append(
                child_node
            )

        else:

            print(
                f"Unable to uniquely identify parent "
                f"for userId={child_row['userId']} "
                f"pid={child_row['pid']}"
            )

            roots.append(child_node)


# save pretty JSON
with open(
    "Top4_hierarchy.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        roots,
        f,
        ensure_ascii=False,
        indent=2
    )


print(
    f"Created {len(roots)} root nodes"
)

Created 3 root nodes
